# CityCare Diagnostic Assistant

### AI-Powered Lab Test Booking System

**Name :** Utkarsh Pateriya

**Roll No:** SCFP125048

**LLM:** Groq - openai/gpt-oss-20b

**Web Search:** Tavily API

**Database:** SQLite

**Environment:** Google Colab

This project implements an AI-powered diagnostic laboratory assistant
that allows patients to browse available tests, check appointment
slots, book tests, view bookings and cancel bookings.

The assistant also provides general health information using web search.

All booking operations require user approval.
The assistant does not diagnose medical conditions.

In [1]:
%pip install -q -U langchain langchain-groq langchain-tavily langgraph pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


In [2]:
import os
import re
import json
import sqlite3
import uuid
import pandas as pd

from datetime import datetime, timedelta
from typing import TypedDict

from IPython.display import display

# Load API keys from Google Colab Secrets

from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

if not GROQ_API_KEY or not TAVILY_API_KEY:
    raise ValueError("Please configure both API keys in Colab Secrets.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

print("API keys loaded successfully.")

# Import LangChain components

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

from langgraph.checkpoint.memory import InMemorySaver

# Initialize Groq model

MODEL_NAME = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0
)

# Initialize Tavily search

web_search = TavilySearch(
    max_results=4
)

print("Groq model:", MODEL_NAME)
print("Tavily search initialized.")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


API keys loaded successfully.
Groq model: openai/gpt-oss-20b
Tavily search initialized.


In [3]:
DB_PATH = "/content/citycare.db"

conn = sqlite3.connect(
    DB_PATH,
    check_same_thread=False
)

conn.row_factory = sqlite3.Row

conn.execute("PRAGMA foreign_keys = ON")

# Create database tables

conn.executescript("""

CREATE TABLE IF NOT EXISTS tests (

    test_id TEXT PRIMARY KEY,
    test_name TEXT NOT NULL,
    category TEXT NOT NULL,
    description TEXT NOT NULL,
    price REAL NOT NULL

);

CREATE TABLE IF NOT EXISTS slots (

    slot_id TEXT PRIMARY KEY,
    test_id TEXT NOT NULL,
    slot_date TEXT NOT NULL,
    slot_time TEXT NOT NULL,
    status TEXT NOT NULL,

    FOREIGN KEY(test_id)
    REFERENCES tests(test_id)

);

CREATE TABLE IF NOT EXISTS bookings (

    booking_id TEXT PRIMARY KEY,
    user_id TEXT NOT NULL,
    test_id TEXT NOT NULL,
    slot_id TEXT NOT NULL UNIQUE,
    status TEXT NOT NULL,
    booking_date TEXT NOT NULL,

    FOREIGN KEY(test_id)
    REFERENCES tests(test_id),

    FOREIGN KEY(slot_id)
    REFERENCES slots(slot_id)

);

""")

print("SQLite database created successfully.")

SQLite database created successfully.


In [4]:
# CityCare diagnostic test catalogue

test_data = [

    ("CT101", "Complete Blood Count", "Blood",
     "Measures different components of blood.", 450),

    ("CT102", "HbA1c Test", "Blood",
     "Measures average blood glucose levels.", 680),

    ("CT103", "Vitamin B12 Test", "Blood",
     "Measures vitamin B12 levels in the blood.", 750),

    ("CT201", "Urine Routine Test", "Urine",
     "Examines physical and chemical properties of urine.", 280),

    ("CT202", "Urine Culture Test", "Urine",
     "Checks urine for bacterial growth.", 590),

    ("CT301", "Chest X-Ray", "Imaging",
     "Produces an image of the chest.", 800),

    ("CT302", "Abdominal Ultrasound", "Imaging",
     "Uses ultrasound to examine abdominal organs.", 1250),

    ("CT401", "Electrocardiogram", "Cardiac",
     "Records the electrical activity of the heart.", 400)

]

conn.executemany("""

INSERT OR IGNORE INTO tests
(test_id, test_name, category, description, price)

VALUES (?, ?, ?, ?, ?)

""", test_data)

conn.commit()

print("CityCare test records inserted.")

CityCare test records inserted.


In [5]:
# Generate appointment slots

start_date = datetime.now().date() + timedelta(days=1)

slot_data = []

for test_id, *_ in test_data:

    for i, time_slot in enumerate(
        ["09:00", "11:00", "14:00"], 1
    ):

        slot_id = f"{test_id}-S{i}"

        slot_date = (
            start_date + timedelta(days=i-1)
        ).isoformat()

        # ECG has no available slots

        if test_id == "CT401":
            status = "Booked"
        else:
            status = "Available"

        slot_data.append((
            slot_id,
            test_id,
            slot_date,
            time_slot,
            status
        ))

conn.executemany("""

INSERT OR IGNORE INTO slots
(slot_id, test_id, slot_date, slot_time, status)

VALUES (?, ?, ?, ?, ?)

""", slot_data)

# Create sample bookings for the fully booked ECG test

for i in range(1, 4):

    conn.execute("""

    INSERT OR IGNORE INTO bookings
    (booking_id, user_id, test_id, slot_id, status, booking_date)

    VALUES (?, ?, ?, ?, ?, ?)

    """, (
        f"DEMO-ECG-{i}",
        "PATIENT_DEMO",
        "CT401",
        f"CT401-S{i}",
        "Confirmed",
        datetime.now().date().isoformat()
    ))

conn.commit()

print("Appointment slots inserted successfully.")

Appointment slots inserted successfully.


In [6]:
def display_table(table_name):

    df = pd.read_sql_query(
        f"SELECT * FROM {table_name}",
        conn
    )

    display(df)


print("========== TEST CATALOGUE ==========")

display_table("tests")

print("========== APPOINTMENT SLOTS ==========")

display_table("slots")

print("========== BOOKINGS ==========")

display_table("bookings")

========== TEST CATALOGUE ==========


,test_id,test_name,category,description,price
0,CT101,Complete Blood Count,Blood,Measures different components of blood.,450.0
1,CT102,HbA1c Test,Blood,Measures average blood glucose levels.,680.0
2,CT103,Vitamin B12 Test,Blood,Measures vitamin B12 levels in the blood.,750.0
3,CT201,Urine Routine Test,Urine,Examines physical and chemical properties of u...,280.0
4,CT202,Urine Culture Test,Urine,Checks urine for bacterial growth.,590.0
5,CT301,Chest X-Ray,Imaging,Produces an image of the chest.,800.0
6,CT302,Abdominal Ultrasound,Imaging,Uses ultrasound to examine abdominal organs.,1250.0
7,CT401,Electrocardiogram,Cardiac,Records the electrical activity of the heart.,400.0


========== APPOINTMENT SLOTS ==========


,slot_id,test_id,slot_date,slot_time,status
0,CT101-S1,CT101,2026-11-21,09:00,Available
1,CT101-S2,CT101,2026-11-22,11:00,Available
2,CT101-S3,CT101,2026-11-23,14:00,Available
3,CT102-S1,CT102,2026-11-21,09:00,Available
4,CT102-S2,CT102,2026-11-22,11:00,Available
5,CT102-S3,CT102,2026-11-23,14:00,Available
6,CT103-S1,CT103,2026-11-21,09:00,Available
7,CT103-S2,CT103,2026-11-22,11:00,Available
8,CT103-S3,CT103,2026-11-23,14:00,Available
9,CT201-S1,CT201,2026-11-21,09:00,Available


========== BOOKINGS ==========


,booking_id,user_id,test_id,slot_id,status,booking_date
0,DEMO-ECG-1,PATIENT_DEMO,CT401,CT401-S1,Confirmed,2026-11-20
1,DEMO-ECG-2,PATIENT_DEMO,CT401,CT401-S2,Confirmed,2026-11-20
2,DEMO-ECG-3,PATIENT_DEMO,CT401,CT401-S3,Confirmed,2026-11-20


In [7]:
# Define patient context

class PatientContext(TypedDict):

    user_id: str


# Default logged-in patient

ACTIVE_USER_ID = "PATIENT_1001"

# Store executed tool names

TOOL_CALLS = []

# Approval setting

APPROVAL_MODE = "interactive"


def record_tool(tool_name):

    TOOL_CALLS.append(tool_name)

    print(f"[TOOL EXECUTED] {tool_name}")


def change_patient(user_id):

    global ACTIVE_USER_ID

    ACTIVE_USER_ID = user_id

    print("Active patient changed to:", user_id)


# Print the database schema visible to the agent

schema = {

    "tests": [
        "test_id",
        "test_name",
        "category",
        "description",
        "price"
    ],

    "slots": [
        "slot_id",
        "test_id",
        "slot_date",
        "slot_time",
        "status"
    ],

    "bookings": [
        "booking_id",
        "user_id",
        "test_id",
        "slot_id",
        "status",
        "booking_date"
    ],

    "user_identity": "Provided through runtime context"

}

print(json.dumps(schema, indent=4))

{
    "tests": [
        "test_id",
        "test_name",
        "category",
        "description",
        "price"
    ],
    "slots": [
        "slot_id",
        "test_id",
        "slot_date",
        "slot_time",
        "status"
    ],
    "bookings": [
        "booking_id",
        "user_id",
        "test_id",
        "slot_id",
        "status",
        "booking_date"
    ],
    "user_identity": "Provided through runtime context"
}


In [8]:
# Function to list all available diagnostic tests

def get_all_tests():

    rows = conn.execute("""

    SELECT test_id, test_name, category,
           description, price

    FROM tests

    ORDER BY category

    """).fetchall()

    return [dict(row) for row in rows]


# Function to check appointment availability

def get_available_slots(test_id):

    rows = conn.execute("""

    SELECT slot_id, slot_date, slot_time

    FROM slots

    WHERE test_id = ?
    AND status = 'Available'

    ORDER BY slot_date, slot_time

    """, (test_id,)).fetchall()

    return [dict(row) for row in rows]


# Function to retrieve bookings of a particular patient

def get_patient_bookings(user_id):

    rows = conn.execute("""

    SELECT
        b.booking_id,
        t.test_name,
        s.slot_date,
        s.slot_time,
        b.status

    FROM bookings b

    JOIN tests t
    ON b.test_id = t.test_id

    JOIN slots s
    ON b.slot_id = s.slot_id

    WHERE b.user_id = ?
    AND b.status = 'Confirmed'

    """, (user_id,)).fetchall()

    return [dict(row) for row in rows]


print("Database read functions created.")

Database read functions created.


In [11]:
# Initialize retry simulation variable

FAIL_NEXT_SLOT_CHECK_ONCE = False
@tool
def list_tests():
    """Display all diagnostic tests, their categories and prices."""

    record_tool("list_tests")

    tests = get_all_tests()

    return json.dumps(tests, indent=2)


@tool
def check_slots(test_id: str):
    """Check available appointment slots for a given test ID."""

    record_tool("check_slots")

    for attempt in range(2):

        try:

            global FAIL_NEXT_SLOT_CHECK_ONCE

            if FAIL_NEXT_SLOT_CHECK_ONCE:

                FAIL_NEXT_SLOT_CHECK_ONCE = False

                raise RuntimeError(
                    "Temporary database connection failure"
                )

            slots = get_available_slots(test_id)

            if not slots:

                return "No available slots for this test."

            return json.dumps(slots, indent=2)

        except RuntimeError as error:

            if attempt == 0:

                print("[RETRY] Retrying slot lookup...")

                continue

            return "Slot lookup failed after retry."


@tool
def my_bookings(runtime: ToolRuntime[PatientContext]):
    """Display confirmed bookings of the currently logged-in patient."""

    record_tool("my_bookings")

    user_id = runtime.context["user_id"]

    bookings = get_patient_bookings(user_id)

    if not bookings:

        return "No confirmed bookings found."

    return json.dumps(bookings, indent=2)


print("Three read tools registered.")

Three read tools registered.


In [12]:
print("========== AVAILABLE TESTS ==========")

print(list_tests.invoke({}))


print("\n========== AVAILABLE SLOTS ==========")

print(check_slots.invoke({
    "test_id": "CT101"
}))


print("\n========== FULLY BOOKED TEST ==========")

print(check_slots.invoke({
    "test_id": "CT401"
}))

========== AVAILABLE TESTS ==========
[TOOL EXECUTED] list_tests
[
  {
    "test_id": "CT101",
    "test_name": "Complete Blood Count",
    "category": "Blood",
    "description": "Measures different components of blood.",
    "price": 450.0
  },
  {
    "test_id": "CT102",
    "test_name": "HbA1c Test",
    "category": "Blood",
    "description": "Measures average blood glucose levels.",
    "price": 680.0
  },
  {
    "test_id": "CT103",
    "test_name": "Vitamin B12 Test",
    "category": "Blood",
    "description": "Measures vitamin B12 levels in the blood.",
    "price": 750.0
  },
  {
    "test_id": "CT401",
    "test_name": "Electrocardiogram",
    "category": "Cardiac",
    "description": "Records the electrical activity of the heart.",
    "price": 400.0
  },
  {
    "test_id": "CT301",
    "test_name": "Chest X-Ray",
    "category": "Imaging",
    "description": "Produces an image of the chest.",
    "price": 800.0
  },
  {
    "test_id": "CT302",
    "test_name": "Abdominal 

In [ ]:
def request_approval(action, details):

    print("\n========== APPROVAL REQUIRED ==========")

    print("Action:", action)

    print("Details:", details)

    if APPROVAL_MODE == "approve":

        decision = "yes"

        print("Automatic demonstration: APPROVED")

    elif APPROVAL_MODE == "reject":

        decision = "no"

        print("Automatic demonstration: REJECTED")

    else:

        decision = input(
            "Do you approve this operation? (yes/no): "
        ).strip().lower()

    return decision in ["yes", "y", "approve"]


print("Approval mechanism initialized.")

In [13]:
def create_booking(test_id, slot_id, user_id, approved=None):

    record_tool("book_test")

    # Validate test

    test = conn.execute("""

    SELECT test_name, price

    FROM tests

    WHERE test_id = ?

    """, (test_id,)).fetchone()

    if not test:

        return {
            "success": False,
            "message": "Invalid test ID."
        }

    # Validate appointment slot

    slot = conn.execute("""

    SELECT slot_date, slot_time, status

    FROM slots

    WHERE slot_id = ?
    AND test_id = ?

    """, (slot_id, test_id)).fetchone()

    if not slot:

        return {
            "success": False,
            "message": "Invalid appointment slot."
        }

    if slot["status"] != "Available":

        return {
            "success": False,
            "message": "Appointment slot is already booked."
        }

    # Check duplicate booking

    duplicate = conn.execute("""

    SELECT booking_id

    FROM bookings

    WHERE user_id = ?
    AND test_id = ?
    AND status = 'Confirmed'

    """, (user_id, test_id)).fetchone()

    if duplicate:

        return {
            "success": False,
            "message": "You already have a confirmed booking for this test."
        }

    # Request approval

    details = {
        "test": test["test_name"],
        "price": test["price"],
        "date": slot["slot_date"],
        "time": slot["slot_time"]
    }

    if approved is None:

        approved = request_approval(
            "Book Diagnostic Test",
            details
        )

    if not approved:

        return {
            "success": False,
            "message": "Booking rejected. Database unchanged."
        }

    # Generate booking ID

    booking_id = "CC-" + uuid.uuid4().hex[:8].upper()

    try:

        conn.execute("BEGIN IMMEDIATE")

        # Recheck slot before updating

        current_slot = conn.execute("""

        SELECT status

        FROM slots

        WHERE slot_id = ?

        """, (slot_id,)).fetchone()

        if not current_slot or current_slot["status"] != "Available":

            conn.rollback()

            return {
                "success": False,
                "message": "Slot is no longer available."
            }

        # Update slot

        conn.execute("""

        UPDATE slots

        SET status = 'Booked'

        WHERE slot_id = ?

        """, (slot_id,))

        # Insert booking

        conn.execute("""

        INSERT INTO bookings

        VALUES (?, ?, ?, ?, ?, ?)

        """, (
            booking_id,
            user_id,
            test_id,
            slot_id,
            "Confirmed",
            datetime.now().date().isoformat()
        ))

        conn.commit()

    except Exception:

        conn.rollback()

        raise

    # Generate receipt directly from database

    receipt = conn.execute("""

    SELECT
        b.booking_id,
        b.user_id,
        t.test_name,
        t.price,
        s.slot_date,
        s.slot_time,
        b.status

    FROM bookings b

    JOIN tests t
    ON b.test_id = t.test_id

    JOIN slots s
    ON b.slot_id = s.slot_id

    WHERE b.booking_id = ?

    """, (booking_id,)).fetchone()

    print("\n========== BOOKING RECEIPT ==========")

    print(json.dumps(dict(receipt), indent=4))

    return {
        "success": True,
        "message": "Booking successfully confirmed.",
        "receipt": dict(receipt)
    }


print("Booking function created.")

Booking function created.


In [14]:
def cancel_patient_booking(booking_id, user_id, approved=None):

    record_tool("cancel_booking")

    # Check booking ownership

    booking = conn.execute("""

    SELECT
        b.booking_id,
        b.slot_id,
        t.test_name,
        s.slot_date,
        s.slot_time

    FROM bookings b

    JOIN tests t
    ON b.test_id = t.test_id

    JOIN slots s
    ON b.slot_id = s.slot_id

    WHERE b.booking_id = ?
    AND b.user_id = ?
    AND b.status = 'Confirmed'

    """, (booking_id, user_id)).fetchone()

    if not booking:

        return {
            "success": False,
            "message": "Booking not found for this patient."
        }

    # Request approval

    if approved is None:

        approved = request_approval(
            "Cancel Diagnostic Test",
            dict(booking)
        )

    if not approved:

        return {
            "success": False,
            "message": "Cancellation rejected. Database unchanged."
        }

    try:

        conn.execute("BEGIN IMMEDIATE")

        # Cancel booking

        conn.execute("""

        UPDATE bookings

        SET status = 'Cancelled'

        WHERE booking_id = ?
        AND user_id = ?

        """, (booking_id, user_id))

        # Release appointment slot

        conn.execute("""

        UPDATE slots

        SET status = 'Available'

        WHERE slot_id = ?

        """, (booking["slot_id"],))

        conn.commit()

    except Exception:

        conn.rollback()

        raise

    return {
        "success": True,
        "message": "Booking cancelled successfully."
    }


print("Cancellation function created.")

Cancellation function created.


In [15]:
@tool
def book_test(
    test_id: str,
    slot_id: str,
    runtime: ToolRuntime[PatientContext]
):
    """Book a selected diagnostic test after patient approval."""

    user_id = runtime.context["user_id"]

    result = create_booking(
        test_id,
        slot_id,
        user_id
    )

    return json.dumps(result, default=str)


@tool
def cancel_booking(
    booking_id: str,
    runtime: ToolRuntime[PatientContext]
):
    """Cancel an existing booking after patient approval."""

    user_id = runtime.context["user_id"]

    result = cancel_patient_booking(
        booking_id,
        user_id
    )

    return json.dumps(result, default=str)


print("Booking and cancellation tools registered.")

Booking and cancellation tools registered.


In [16]:
# Find an available appointment

available_slot = conn.execute("""

SELECT test_id, slot_id

FROM slots

WHERE status = 'Available'

LIMIT 1

""").fetchone()

# Count bookings before rejection

before = conn.execute("""

SELECT COUNT(*)

FROM bookings

""").fetchone()[0]

# Demonstrate rejection

rejection_result = create_booking(
    available_slot["test_id"],
    available_slot["slot_id"],
    "PATIENT_REJECT",
    approved=False
)

# Count bookings after rejection

after = conn.execute("""

SELECT COUNT(*)

FROM bookings

""").fetchone()[0]

print("Rejection result:", rejection_result)

print("Bookings before:", before)

print("Bookings after:", after)

assert before == after

print("\nPASS: Rejection did not modify the database.")

[TOOL EXECUTED] book_test
Rejection result: {'success': False, 'message': 'Booking rejected. Database unchanged.'}
Bookings before: 3
Bookings after: 3

PASS: Rejection did not modify the database.


In [17]:
# Find a different available appointment

available_slot = conn.execute("""

SELECT test_id, slot_id

FROM slots

WHERE status = 'Available'
AND test_id = 'CT101'

LIMIT 1

""").fetchone()

# Demonstrate approved booking

booking_result = create_booking(
    available_slot["test_id"],
    available_slot["slot_id"],
    "PATIENT_1001",
    approved=True
)

print("\nBooking result:")

print(json.dumps(booking_result, indent=4, default=str))

[TOOL EXECUTED] book_test

========== BOOKING RECEIPT ==========
{
    "booking_id": "CC-9E0AF865",
    "user_id": "PATIENT_1001",
    "test_name": "Complete Blood Count",
    "price": 450.0,
    "slot_date": "2026-11-21",
    "slot_time": "09:00",
    "status": "Confirmed"
}

Booking result:
{
    "success": true,
    "message": "Booking successfully confirmed.",
    "receipt": {
        "booking_id": "CC-9E0AF865",
        "user_id": "PATIENT_1001",
        "test_name": "Complete Blood Count",
        "price": 450.0,
        "slot_date": "2026-11-21",
        "slot_time": "09:00",
        "status": "Confirmed"
    }
}


In [18]:
FAIL_NEXT_SLOT_CHECK_ONCE = False

In [19]:
@tool
def general_health_search(query: str):
    """
    Search general health information such as test purposes,
    fasting requirements and basic test preparation.
    Never search for CityCare prices or appointment slots.
    """

    record_tool("general_health_search")

    if not query.strip():

        return "Please provide a search question."

    results = web_search.invoke({
        "query": query
    })

    return str(results)[:8000]


print("Tavily health information tool registered.")

Tavily health information tool registered.


In [20]:
SYSTEM_PROMPT = """

You are CityCare Diagnostic Assistant.

You help patients interact with a fictional diagnostic laboratory.

YOUR RESPONSIBILITIES:

1. Display diagnostic tests using list_tests.

2. Display prices only from the SQLite database.

3. Check appointment availability using check_slots.

4. Display patient bookings using my_bookings.

5. Book tests using book_test.

6. Cancel bookings using cancel_booking.

7. Use general_health_search for educational health information.

IMPORTANT RULES:

- Never invent test prices.
- Never invent appointment availability.
- Never modify the database without approval.
- Always request approval before booking.
- Always request approval before cancellation.
- Never claim a booking succeeded unless the tool confirms it.
- Never expose another patient's booking information.
- Patient identity comes from runtime context.
- Never ask the language model to supply user_id.
- Never execute raw SQL generated by the model.
- Never diagnose a medical condition.
- Never select or recommend a test based on symptoms.
- For symptom-related questions, explain that you cannot diagnose
  and recommend consulting a qualified healthcare professional.
- For severe or sudden symptoms, advise seeking emergency care.

GENERAL INFORMATION:

Use Tavily only for general educational information.

Prices and appointment availability must come exclusively
from the CityCare SQLite database.

Be polite, professional and concise.

"""
print("System prompt created.")

System prompt created.


In [21]:
# Initialize conversation memory

memory = InMemorySaver()

# Register all available tools

tools = [

    list_tests,

    check_slots,

    my_bookings,

    book_test,

    cancel_booking,

    general_health_search

]

# Create CityCare AI agent

agent = create_agent(

    model=llm,

    tools=tools,

    system_prompt=SYSTEM_PROMPT,

    context_schema=PatientContext,

    checkpointer=memory

)

print("CityCare AI Agent created successfully.")

print("Model:", MODEL_NAME)

print("Registered tools:")

for t in tools:

    print("-", t.name)

CityCare AI Agent created successfully.
Model: openai/gpt-oss-20b
Registered tools:
- list_tests
- check_slots
- my_bookings
- book_test
- cancel_booking
- general_health_search


In [22]:
# Clear tool execution history

TOOL_CALLS.clear()

# Invoke agent

response = agent.invoke(

    {

        "messages": [

            {

                "role": "user",

                "content": "Show me all available diagnostic tests with their prices."

            }

        ]

    },

    context={

        "user_id": "PATIENT_1001"

    },

    config={

        "configurable": {

            "thread_id": "patient_1001"

        }

    }

)

print("Agent response:")

print(response["messages"][-1].content)

print("\nTools executed:")

print(TOOL_CALLS)

[TOOL EXECUTED] list_tests
Agent response:
Here are all the diagnostic tests currently available at CityCare, along with their prices:

| Test ID | Test Name                | Category | Price (₹) |
|---------|--------------------------|----------|-----------|
| CT101   | Complete Blood Count     | Blood    | 450.00    |
| CT102   | HbA1c Test               | Blood    | 680.00    |
| CT103   | Vitamin B12 Test         | Blood    | 750.00    |
| CT401   | Electrocardiogram        | Cardiac  | 400.00    |
| CT301   | Chest X‑Ray              | Imaging  | 800.00    |
| CT302   | Abdominal Ultrasound     | Imaging  | 1,250.00  |
| CT201   | Urine Routine Test       | Urine    | 280.00    |
| CT202   | Urine Culture Test       | Urine    | 590.00    |

Let me know if you’d like to check appointment availability for a specific test or if you’d like to book one.

Tools executed:
['list_tests']


In [24]:
# Email redaction

EMAIL_PATTERN = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

def redact_email(message):

    # Remove email addresses, including those inside Markdown links

    return re.sub(
        EMAIL_PATTERN,
        "[EMAIL REDACTED]",
        message
    )


# Example user input

original_message = (
    "Please send my report to patient.citycare@gmail.com"
)

# Redact email before sending to AI

safe_message = redact_email(original_message)


print("Original message:")
print(original_message)

print("\nMessage received by AI:")
print(safe_message)


# Verify redaction

assert "patient.citycare@gmail.com" not in safe_message

assert "[EMAIL REDACTED]" in safe_message

print("\nSUCCESS: Email successfully redacted!")

Original message:
Please send my report to patient.citycare@gmail.com

Message received by AI:
Please send my report to [EMAIL REDACTED]

SUCCESS: Email successfully redacted!


In [25]:
markdown_message = (
    "Please send my report to "
    "[patient.citycare@gmail.com](mailto:patient.citycare@gmail.com)"
)

redacted_message = redact_email(markdown_message)

print("Before:", markdown_message)

print("\nAfter:", redacted_message)

assert "patient.citycare@gmail.com" not in redacted_message

print("\nPASS: Markdown email redaction verified.")

Before: Please send my report to [patient.citycare@gmail.com](mailto:patient.citycare@gmail.com)

After: Please send my report to [[EMAIL REDACTED]](mailto:[EMAIL REDACTED])

PASS: Markdown email redaction verified.


In [26]:
# Force the next slot check to fail once

FAIL_NEXT_SLOT_CHECK_ONCE = True

TOOL_CALLS.clear()

retry_response = agent.invoke(

    {

        "messages": [

            {

                "role": "user",

                "content": "Check the available appointment slots for test CT102."

            }

        ]

    },

    context={

        "user_id": "PATIENT_1001"

    },

    config={

        "configurable": {

            "thread_id": "retry_demo"

        }

    }

)

print("\nAgent response after retry:")

print(retry_response["messages"][-1].content)

print("\nTool execution history:")

print(TOOL_CALLS)

[TOOL EXECUTED] check_slots
[RETRY] Retrying slot lookup...

Agent response after retry:
Here are the available appointment slots for **CT102**:

| Slot ID | Date       | Time |
|---------|------------|------|
| CT102-S1 | 2026‑11‑21 | 09:00 |
| CT102-S2 | 2026‑11‑22 | 11:00 |
| CT102-S3 | 2026‑11‑23 | 14:00 |

Let me know if you’d like to book one of these slots or if you need anything else!

Tool execution history:
['check_slots']


In [27]:
def run_chat():

    global ACTIVE_USER_ID

    print("\n====================================")

    print("WELCOME TO CITYCARE DIAGNOSTIC ASSISTANT")

    print("====================================")

    print("Available commands:")

    print("/patient PATIENT_1001")

    print("/patient PATIENT_2002")

    print("exit")

    while True:

        user_message = input("\nYou: ").strip()

        if user_message.lower() in ["exit", "quit"]:

            print("Thank you for using CityCare.")

            break

        # Switch patient

        if user_message.startswith("/patient "):

            new_user = user_message.split(maxsplit=1)[1]

            if not re.fullmatch(
                r"PATIENT_[0-9]{4}",
                new_user
            ):

                print("Invalid demo patient ID.")

                continue

            change_patient(new_user)

            continue

        # Remove email addresses before model invocation

        safe_message = redact_email(user_message)

        # Clear previous tool history

        TOOL_CALLS.clear()

        try:

            response = agent.invoke(

                {

                    "messages": [

                        {

                            "role": "user",

                            "content": safe_message

                        }

                    ]

                },

                context={

                    "user_id": ACTIVE_USER_ID

                },

                config={

                    "configurable": {

                        "thread_id": ACTIVE_USER_ID

                    }

                }

            )

            print("\nTools executed:")

            print(TOOL_CALLS)

            print("\nCityCare Assistant:")

            print(response["messages"][-1].content)

        except Exception as error:

            print("An error occurred:", error)


print("Interactive chatbot is ready.")

Interactive chatbot is ready.


In [28]:
run_chat()


WELCOME TO CITYCARE DIAGNOSTIC ASSISTANT
Available commands:
/patient PATIENT_1001
/patient PATIENT_2002
exit

You: hii

Tools executed:
[]

CityCare Assistant:
Hello! 👋 How can I assist you with your CityCare diagnostic needs today?

You: show all tests
[TOOL EXECUTED] list_tests

Tools executed:
['list_tests']

CityCare Assistant:
Here’s a list of all the diagnostic tests we offer, along with their categories, brief descriptions, and prices:

| Test ID | Test Name | Category | Description | Price (₹) |
|---------|-----------|----------|-------------|-----------|
| CT101 | Complete Blood Count | Blood | Measures different components of blood. | 450 |
| CT102 | HbA1c Test | Blood | Measures average blood glucose levels. | 680 |
| CT103 | Vitamin B12 Test | Blood | Measures vitamin B12 levels in the blood. | 750 |
| CT401 | Electrocardiogram | Cardiac | Records the electrical activity of the heart. | 400 |
| CT301 | Chest X‑Ray | Imaging | Produces an image of the chest. | 800 |
| CT30